# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [ ]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

**Check the shape of your data**

In [ ]:
#your code here
spaceship.shape

**Check for data types**

In [ ]:
#your code here
spaceship.info()

**Check for missing values**

In [ ]:
#your code here
spaceship.isnull().sum()

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [ ]:
#your code here
spaceship.dropna(inplace=True)

In [ ]:
spaceship.isnull().sum()

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [ ]:
#your code here
spaceship["Cabin"].value_counts()
spaceship['Cabin'] = spaceship['Cabin'].str.split('/').str[0]
spaceship['Cabin'].unique()

- Drop PassengerId and Name

In [ ]:
#your code here
spaceship = spaceship.drop(['PassengerId', 'Name'], axis=1)


- For non-numerical columns, do dummies.

In [ ]:
#your code here
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
spaceship.info()

In [ ]:
spaceship.HomePlanet.unique()

In [ ]:
from sklearn.preprocessing import OneHotEncoder



In [ ]:

# 1. Initialize the encoder
encoder = OneHotEncoder(drop='first', sparse_output=False)

# 2. Fit and transform the column
encoded = encoder.fit_transform(spaceship[['HomePlanet']])

# 3. Create a DataFrame from the encoded array
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['HomePlanet'])
)

# 4. Concatenate with original DataFrame (or drop original if needed)
spaceship = pd.concat([spaceship.drop(columns=['HomePlanet']), encoded_df], axis=1)

#print(spaceship)


In [ ]:

spaceship.head()


In [ ]:
spaceship.VIP.unique()


**Perform Train Test Split**

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# List of categorical columns to encode
categorical_cols = ['CryoSleep', 'Cabin', 'Destination', 'VIP']

# Drop rows with missing values in categorical columns
spaceship_cleaned = spaceship.dropna(subset=categorical_cols)

# OneHotEncoder with drop='first' and handling unknown categories
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# ColumnTransformer to apply encoding only to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols)
    ],
    remainder='passthrough'  # Leave other columns (like Age) untouched
)

# Transform the data
processed = preprocessor.fit_transform(spaceship_cleaned)

# Get column names after encoding
encoded_columns = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_columns = list(encoded_columns) + [col for col in spaceship_cleaned.columns if col not in categorical_cols]

# Create final DataFrame
spaceship_encoded = pd.DataFrame(processed, columns=all_columns)

print(spaceship_encoded)

In [ ]:
#your code here
spaceship_encoded.info()

**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [ ]:
y = spaceship_encoded['Transported'].astype(bool)
print(y_train.dtype)
print(y_train.unique())

In [ ]:
#your code here
# Separate features and target
X = spaceship_encoded.drop(columns='Transported')
y = spaceship_encoded['Transported']  # Keep as boolean

# Convert all X columns to numeric
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()
y = y[X.index]  # Align y with the new X


In [ ]:
print(X.isna().sum())


In [ ]:
from sklearn.model_selection import train_test_split

# Split into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on training data and transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


- Evaluate your model's performance. Comment it

In [ ]:
y_train = y_train.astype(bool)
y_test = y_test.astype(bool)


In [ ]:
#your code here
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Predict
y_pred = knn.predict(X_test_scaled)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))
